# Course 6: From Beginner to an AI Research Lab
## Session 4: Machine learning foundations

**Goal:** write logistic regression from scratch, use the three data splits correctly, and read under- and overfitting from learning curves.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. Split first, then standardize
60% training, 20% validation, 20% test, stratified by label. The scaler is fitted on the training set only.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.25, random_state=0, stratify=y_tr)
scaler = StandardScaler().fit(X_tr)
X_tr, X_va, X_te = (scaler.transform(a) for a in (X_tr, X_va, X_te))
print("train", X_tr.shape, "validation", X_va.shape, "test", X_te.shape)

### 2. Logistic regression from scratch

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def fit(X, y, lam=0.01, lr=0.1, steps=2000):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)
        w -= lr * (X.T @ (p - y) / len(y) + lam * w)
        b -= lr * np.mean(p - y)
    return w, b

def accuracy(w, b, X, y):
    return np.mean((sigmoid(X @ w + b) > 0.5) == y)

w, b = fit(X_tr, y_tr)
acc_va, acc_te = accuracy(w, b, X_va, y_va), accuracy(w, b, X_te, y_te)
print(f"validation accuracy {acc_va:.3f} | test accuracy {acc_te:.3f}")
assert abs(acc_va - 0.974) < 0.01 and abs(acc_te - 0.974) < 0.01

### 3. Compare with scikit-learn
scikit-learn's `C` is the inverse of the regularization strength, scaled by the number of rows.

In [ ]:
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(C=1 / (0.01 * len(y_tr)), max_iter=5000).fit(X_tr, y_tr)
print(f"scikit-learn: validation {sk.score(X_va, y_va):.3f} | test {sk.score(X_te, y_te):.3f}")
assert abs(sk.score(X_te, y_te) - acc_te) < 0.02

### 4. Choose the regularization strength on the validation set

In [ ]:
sweep = {}
for lam in [0, 0.001, 0.01, 0.1, 1]:
    wl, bl = fit(X_tr, y_tr, lam=lam); sweep[lam] = (accuracy(wl, bl, X_tr, y_tr), accuracy(wl, bl, X_va, y_va))
    print(f"lam {lam:<6} train accuracy {sweep[lam][0]:.3f}   validation accuracy {sweep[lam][1]:.3f}")
best_lam = max(sweep, key=lambda k: sweep[k][1])
print("chosen on validation:", best_lam)
assert sweep[1][1] < sweep[0.001][1]

Each validation or test set has 114 rows, so one row is worth about 0.9 points of accuracy. Small differences in this sweep are noise. Session 9 shows how to measure that.

### 5. Learning curves
Train on growing parts of the training set and plot both accuracies.

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0); order = rng.permutation(len(y_tr)); sizes = [int(f * len(y_tr)) for f in np.linspace(0.1, 1.0, 10)]
tr_acc, va_acc = [], []
for n in sizes:
    idx = order[:n]; wn, bn = fit(X_tr[idx], y_tr[idx], lam=best_lam)
    tr_acc.append(accuracy(wn, bn, X_tr[idx], y_tr[idx])); va_acc.append(accuracy(wn, bn, X_va, y_va))
plt.plot(sizes, tr_acc, "o-", label="training"); plt.plot(sizes, va_acc, "s-", label="validation"); plt.xlabel("training rows"); plt.ylabel("accuracy"); plt.legend(); plt.savefig("learning_curve.png")
for n, a, v in zip(sizes, tr_acc, va_acc): print(f"{n:4d} rows   train {a:.3f}   validation {v:.3f}   gap {a - v:+.3f}")

How to read it:
- A large gap that closes as data grows: the model overfits, and more data helps.
- Both curves low and close together: the model underfits, and more data will not help; a richer model might.
- Both high and close: there is little left to gain on this dataset.

### 6. Report the test score once

In [ ]:
w, b = fit(X_tr, y_tr, lam=best_lam)
print(f"final: lam = {best_lam}, test accuracy = {accuracy(w, b, X_te, y_te):.3f} on {len(y_te)} rows")

### 7. The leak to avoid
Fitting the scaler on all the data before splitting lets the test rows influence training. On this dataset the effect is small, and that is the danger: it does not announce itself.

In [ ]:
Xall = StandardScaler().fit_transform(X)                                           # WRONG: the scaler has seen the test rows
A_tr, A_te, b_tr, b_te = train_test_split(Xall, y, test_size=0.2, random_state=0, stratify=y)
wl, bl = fit(A_tr, b_tr, lam=best_lam)
print(f"with the leaky scaler: test accuracy {accuracy(wl, bl, A_te, b_te):.3f} (the number may barely move, and it is still not a clean estimate)")

### TODO
1. Add squared features and repeat the sweep. Does the model now overfit at lam = 0?
2. Repeat the whole notebook with 5 different `random_state` values for the split. How much does the test accuracy vary?
3. Explain in three sentences why the test set is used once.